# Assignment 1
### Title - Parallel Searching Algorithms
Problem Statement - Design and implement Parallel Breadth First Search and Depth
First Search based on existing algorithms using OpenMP. Use a
Tree or an undirected graph for BFS and DFS .

The ```%%writefile``` is used to create a file and then write into it (you can see afile created in the folders)
##### in case you code and want to run it in terminal then avaoid the first line

In [4]:
%%writefile parallel_graph.cpp
#include <iostream>                                                             // For input and output
#include <vector>                                                               // For using std::vector as adjacency list
#include <queue>                                                                // For BFS queue
#include <omp.h>                                                                // OpenMP header for parallel processing

using namespace std;

                                                                                // Graph class using adjacency list representation
class Graph {
    int V;                                                                      // Number of vertices
    vector<vector<int>> adj;                                                    // Adjacency list

public:
                                                                                // Constructor to initialize graph with V vertices
    Graph(int V) : V(V), adj(V) {}

                                                                                // Function to add a directed edge from v to w
    void addEdge(int v, int w) {
        adj[v].push_back(w);
    }

private:
                                                                                // Utility function for parallel DFS using OpenMP
    void parallelDFSUtil(int v, vector<bool>& visited) {
        visited[v] = true;
        cout << v << " ";

                                                                                // Parallelize traversal of adjacent vertices
        #pragma omp parallel for
        for (int i = 0; i < static_cast<int>(adj[v].size()); ++i) {
            int n = adj[v][i];
            bool localVisited = false;

                                                                                // Ensure that only one thread checks/updates visited[n] at a time
            #pragma omp critical
            {
                if (!visited[n]) {
                    visited[n] = true;
                    localVisited = true;
                }
            }

                                                                                // Recurse only if this thread successfully marked the node
            if (localVisited) {
                parallelDFSUtil(n, visited);
            }
        }
    }

public:
                                                                                // Function to start parallel DFS from a starting vertex
    void parallelDFS(int startVertex) {
        vector<bool> visited(V, false);                                         // Track visited nodes

                                                                                // Parallel region begins
        #pragma omp parallel
        {
                                                                                // Only one thread should initiate the recursion
            #pragma omp single
            {
                parallelDFSUtil(startVertex, visited);
            }
        }
    }

                                                                                // Function to perform parallel BFS using OpenMP
    void parallelBFS(int startVertex) {
        vector<bool> visited(V, false);                                         // Track visited nodes
        queue<int> q;

        visited[startVertex] = true;
        q.push(startVertex);

                                                                                // Standard BFS loop
        while (!q.empty()) {
            int v;

                                                                                // Ensure only one thread modifies queue at a time
            #pragma omp critical
            {
                v = q.front();
                q.pop();
            }

            cout << v << " ";

                                                                                // Parallelize neighbor processing
            #pragma omp parallel for
            for (int i = 0; i < static_cast<int>(adj[v].size()); ++i) {
                int n = adj[v][i];
                bool localVisited = false;

                                                                                // Synchronize access to the shared visited array and queue
                #pragma omp critical
                {
                    if (!visited[n]) {
                        visited[n] = true;
                        q.push(n);
                        localVisited = true;
                    }
                }
            }
        }
    }
};

int main() {
    int V, E;                                                                   // V = number of vertices, E = number of edges

                                                                                // Read number of vertices and edges from user
    cout << "Enter the number of vertices: ";
    cin >> V;
    cout << "Enter the number of edges: ";
    cin >> E;

    Graph g(V);                                                                 // Create a graph with V vertices

                                                                                // Read all edges from user
    cout << "Enter the edges (format: vertex1 vertex2):\n";
    for (int i = 0; i < E; ++i) {
        int v, w;
        cin >> v >> w;
        g.addEdge(v, w);                                                        // Add directed edge v -> w
    }

    int startVertex;
                                                                                // Read starting vertex for DFS and BFS
    cout << "Enter the starting vertex for DFS and BFS: ";
    cin >> startVertex;

                                                                                // Perform and display DFS
    cout << "Depth-First Search (DFS): ";
    g.parallelDFS(startVertex);
    cout << "\n";

                                                                                // Perform and display BFS
    cout << "Breadth-First Search (BFS): ";
    g.parallelBFS(startVertex);
    cout << "\n";

    return 0;
}


Overwriting parallel_graph.cpp


```parallel_graph.cpp``` file is created \\
and now to run it we use ```!g++``` \\
```-fopenmp``` compiles the code with the openMP mode \\
The object file is created now from it named as ```parallel_graph.o```

In [5]:
!g++ -fopenmp -o parallel_graph parallel_graph.cpp

to run normally use the \\
```!./parallel_graph```

but we have given predefined inputs as
```
Enter the number of vertices: 4
Enter the number of edges: 4 \\
Enter the edges (format: vertex1 vertex2):
0 1
0 2
1 2
2 3
Enter the starting vertex for DFS and BFS: 0
```

In [6]:
!echo -e "4\n4\n0 1\n0 2\n1 2\n2 3\n0" | ./parallel_graph

Enter the number of vertices: Enter the number of edges: Enter the edges (format: vertex1 vertex2):
Enter the starting vertex for DFS and BFS: Depth-First Search (DFS): 0 1 2 3 
Breadth-First Search (BFS): 0 2 1 3 


you can try manually data entry below :

In [7]:
!./parallel_graph

Enter the number of vertices: ^C
